<a href="https://colab.research.google.com/github/xoclonholdings/ZedAI/blob/main/ZedAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZED AI Temporary Colab Bridge

Use this notebook as the temporary inference host for ZED.

Recommended runtime:
- `GPU`
- `T4` if available

Run the cells in order. After the final cell finishes, copy the printed `PUBLIC_URL` into Render as:
- `REMOTE_INFERENCE_URL=<PUBLIC_URL>`
- `REMOTE_INFERENCE_MODE=colab`

Important:
- keep the notebook open while using ZED
- rotate your ngrok auth token if it was ever exposed


In [ ]:
!pip -q install pyngrok fastapi uvicorn transformers accelerate torch sentencepiece

In [ ]:
NGROK_AUTH_TOKEN = "3CGHDGvntsuJQorcVTx8ZEI1rpw_5zPoStsPeidwvMWRXNHch"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000

In [ ]:
from pyngrok import ngrok
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import socket
import time
import torch
import threading
import uvicorn
import requests

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

def is_port_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex(("127.0.0.1", port)) == 0

if "tokenizer" not in globals() or "model" not in globals():
    print("Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        device_map="auto"
    )
else:
    print("Model already loaded.")

if "app" not in globals():
    app = FastAPI()

    class ChatRequest(BaseModel):
        message: str
        system_prompt: str | None = None

    @app.get("/health")
    def health():
        return {"ok": True, "model": MODEL_NAME}

    @app.post("/chat")
    def chat(req: ChatRequest):
        messages = []
        if req.system_prompt:
            messages.append({"role": "system", "content": req.system_prompt})
        messages.append({"role": "user", "content": req.message})

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=250,
                temperature=0.7,
                do_sample=True
            )

        reply = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )
        return {"reply": reply}

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT)

if not is_port_open(PORT):
    print(f"Starting server on port {PORT}...")
    zed_bridge_thread = threading.Thread(target=run_server, daemon=True)
    zed_bridge_thread.start()
    time.sleep(2)
else:
    print(f"Server already running on port {PORT}.")

tunnels = ngrok.get_tunnels()
public_base = None
for tunnel in tunnels:
    if getattr(tunnel, "proto", "") == "https" and f":{PORT}" in getattr(tunnel, "config", {}).get("addr", ""):
        public_base = tunnel.public_url
        break

if not public_base:
    public_base = ngrok.connect(PORT).public_url

print("PUBLIC_URL =", public_base)
print("LOCAL HEALTH =", requests.get(f"http://127.0.0.1:{PORT}/health").text)
print("LOCAL CHAT =", requests.post(
    f"http://127.0.0.1:{PORT}/chat",
    json={"message": "hello"}
).text)
